# Estimating haplotype frequencies from genotypes (2 SNPs)

We can observe the genotypes of two SNPs, but not which alleles sit together on the same chromosome. In this exercise you will build an EM algorithm that estimates the four haplotype frequencies from genotype data alone.

The code is already written for you apart from the key lines, which are marked `????`. Each task says what goes there, and the maths you need is in the text just above it. Work through it in order, the later cells use the functions you write in the earlier ones.

We simulate haplotypes, pair them up into individuals, and then throw the haplotype information away. The genotypes at the first SNP are AA, Aa, aa and at the second SNP BB, Bb, bb.

## Simulate the genotype data

Run the cell below. It sets the true haplotype frequencies and the number of individuals, and simulates the data.

In [ ]:
set.seed(1)

# the TRUE haplotype frequencies. must sum to 1
freqHap <- c(
    freqHapAB = 0.03,
    freqHapaB = 0.20,
    freqHapAb = 0.55,
    freqHapab = 1 -0.03-0.2-0.55
)

# number of individuals to simulate
N <- 1000

## simulate the 2N haplotypes
simHaps <- sample(c("AB","aB","Ab","ab"),2*N,prob=freqHap,replace=T)

# two haplotypes per individual
hapMat <- matrix(simHaps,ncol=2)
cat("hapPairs is the sampled haplotype pairs\n")
table(hapPairs <- paste(hapMat[,1],hapMat[,2],sep="/"))

# genotypes for SNP 1 and SNP 2
SNP1 <- hapMat[,1] %in% c("AB","Ab")  + hapMat[,2] %in% c("AB","Ab") # number of A alleles (0,1,2)
SNP2 <- hapMat[,1] %in% c("AB","aB")  + hapMat[,2] %in% c("AB","aB") # number of B alleles (0,1,2)

# the data you will analyse: counts of the 9 genotype combinations
cat("\nsimulated genotype data as a table:\n")
(genotypeTab <- table(c("aa","Aa","AA")[SNP1+1],c("bb","Bb","BB")[SNP2+1]))

## Task 1: the true allele frequencies

First calculate the true population allele frequencies for the two SNPs from the true haplotype frequencies (`freqHap`) used in the simulation.

Allele A sits on the haplotypes AB and Ab, so its frequency is the sum of those two haplotype frequencies. The same logic gives a, B and b.

In [ ]:
trueFreq <- data.frame(
    freq_A = ????,   # sum of the haplotype frequencies that carry A
    freq_a = ????,
    freq_B = ????,
    freq_b = ????
)
trueFreq

# hint: the entries of freqHap are named, so you can write freqHap["freqHapAB"]

## Task 2: the allele frequencies in the simulated data

Compare the true frequencies with the data by estimating the frequency of allele A and allele B from the genotypes.

`SNP1` holds the number of A alleles for each individual (0, 1 or 2) and `SNP2` the number of B alleles. Each of the N individuals carries 2 alleles per SNP.

In [ ]:
cat("est freq A")
( estFreqA <- ???? )   # total number of A alleles divided by the total number of alleles
cat("\nest freq B")
( estFreqB <- ???? )

# they should be close to freq_A and freq_B from Task 1, but not identical

## Task 3: the haplotype frequencies when the haplotypes are known

Before doing anything hard, calculate the haplotype frequencies from the simulated haplotype pairs. This is the answer the EM algorithm will have to reach later without ever seeing the haplotypes, so keep the numbers.

In [ ]:
cat("table of haplotype pairs\n")
hapPairsTab <- table(hapPairs)
hapPairsTab

## this helper is given to you. It goes from counts of haplotype PAIRS (16 of them)
## to counts of single HAPLOTYPES (4 of them). You will need it again in the M step.
countHaplotypes <- function(hapPairsTab,print=FALSE){
  countHap <- c()
  hapNames <- c("AB","aB","Ab","ab")
  for(x in hapNames){
    pairs <- c(paste(x,hapNames,sep="/"),paste(hapNames,x,sep="/")) # the pair x/x is counted twice
    if(print)
      cat("Haplotype",x,"is in pairs",pairs,"\n")
    countHap[x] <- sum( hapPairsTab[pairs] )
  }
  countHap
}

cat("\nwhich pairs contain which haplotype (note one pair is there twice):\n")
countHap <- countHaplotypes(hapPairsTab,print=TRUE)
cat("\ncounts of the four haplotypes:\n")
countHap

## from counts to frequencies
estFreqHap <- ????

cat("\nestimated haplotype frequencies (from the observed haplotypes)\n")
round(estFreqHap,4)
cat("\ntrue haplotype frequencies\n")
round(freqHap,4)

# Likelihood

Write a likelihood function where the data is the genotypes for both SNPs and the parameter is the haplotype frequencies.

$L(\theta)=p(X|\theta)=\prod_{i=1}^N p(X_i|\theta)$

where $\theta=(\theta_{AB},\theta_{aB},\theta_{Ab},\theta_{ab})$ are the haplotype frequencies and $X$ is the matrix of data where $X_{i,j}\in \{0,1,2\}$ is the genotype for individual $i$ at site $j\in \{1,2\}$. There are $N$ individuals and 2 sites.

We cannot see the haplotypes, only the genotypes, so we introduce a latent state $Z$: the pair of haplotypes $z=(z_1,z_2)$ carried by the individual.

$p(X_i|\theta)=\sum_{z\in\{AB,aB,Ab,ab\}^2}p(X_i|Z=z)p(Z=z|\theta)$

$p(Z=z|\theta)=p(Z_1=z_1|\theta)p(Z_2=z_2|\theta)=\theta_{z_1}\theta_{z_2}$

$p(X_i|Z=z)=0$ if the genotypes are not consistent with the haplotype pair, and 1 otherwise.

### Likelihood for a single individual

$p(X_i|\theta)=\sum_{z\in\{AB,aB,Ab,ab\}^2}p(X_i|Z=z)p(Z=z|\theta)$

The sum runs over all $4\times 4=16$ haplotype pairs. Most of them give $p(X_i|Z=z)=0$, so in practice you add up the few pairs that are consistent with the genotypes.


## Task 4: the likelihood for one individual

Fill in the two missing lines of `likeG`. The loops over the 16 haplotype pairs and the conversion from a haplotype pair to genotypes are given.

In [ ]:
#parameter theta
(theta <- freqHap)

likeG <- function(g1,g2,theta){
  ## input: the two genotypes of one individual (0,1,2 each) and theta
  ## output: p(X_i|theta)

  names(theta) <- c("AB","aB","Ab","ab")

  #p(X_i|theta) = sum_Z p(X|Z)p(Z|theta)
  pX <- 0
  for(z1 in c("AB","aB","Ab","ab"))        # first haplotype
    for(z2 in c("AB","aB","Ab","ab")){     # second haplotype

        # the genotypes that this pair of haplotypes implies
        hapToGeno1 <- z1%in%c("AB","Ab") + z2%in%c("AB","Ab")  # number of A alleles
        hapToGeno2 <- z1%in%c("AB","aB") + z2%in%c("AB","aB")  # number of B alleles

        # this haplotype pair cannot have produced the observed genotypes,
        # so p(X_i|Z=z)=0 and the term adds nothing. Skip it.
        if( ???? )
          next

        # otherwise p(X_i|Z=z)=1, so the term is just p(Z=z|theta)
        pX <- pX + ????
    }
  pX
}

for(g1 in 0:2)
  for(g2 in 0:2)
    cat("like for individuals with genotype SNP1=",g1," SNP2=",g2," is",likeG(g1,g2,theta),"\n")

### Log likelihood of the whole dataset

$L(\theta)=p(X|\theta)=\prod_{i=1}^N p(X_i|\theta)$

We work with the log of it, so the product becomes a sum.


## Task 5: the log likelihood of the whole dataset

Now add up over individuals. One line is missing.

In [ ]:
data <- cbind(SNP1,SNP2)
cat("Data for the first 6 individuals\n")
head(data)

likelihood <- function(data,theta){
  #input: data (N x 2 matrix of genotypes) + theta
  #output: log( p(X|theta) )
  N <- nrow(data)
  logLike <- 0
  for(i in 1:N)
    logLike <- logLike + ????   # the log likelihood of individual i, using likeG

  logLike
}

lik <- likelihood(data,theta)
cat("Log likelihood based on true haplotype frequencies is",lik,"\n")

lik <- likelihood(data,rep(0.25,4))
cat("Log likelihood with uniform haplotype frequencies",lik,"\n")

# which of the two is higher? is that what you expected?

### Faster version

Individuals with the same two genotypes have the same likelihood, so we only need to calculate the individual likelihood for the 9 combinations of genotypes and weight each by how many individuals have it. Those counts are exactly the table `genotypeTab` from the simulation.


## Task 6: the same likelihood, faster

`dataTab` is the 3 x 3 table of genotype counts. One line is missing: the contribution of all individuals that share a genotype combination.

In [ ]:
cat("data\n")
(dataTab <- genotypeTab)
cat("\nparameter theta\n")
(theta <- freqHap)

likelihoodFast <- function(dataTab,theta){
  #input: table of genotype counts (3x3) + theta
  #output: log( p(X|theta) )
  logLike <- 0
  for(g1 in 0:2)
    for(g2 in 0:2){
      likG <- likeG(g1=g1,g2=g2,theta)
      logLike <- logLike + ????   # weight the log likelihood by the number of individuals
    }
 logLike
}

lik <- likelihoodFast(dataTab,theta)
cat("\nLog likelihood based on true haplotype frequencies is",lik,"\n")

lik <- likelihoodFast(dataTab,rep(0.25,4))
cat("\nLog likelihood with uniform haplotype frequencies",lik,"\n")

# both numbers should be identical to the ones from Task 5

# EM algorithm

For the E step we need the posterior probability of the latent haplotype pair,
$q_i(Z=z)=p(Z=z|X_i,\theta^{(n)})$, which is calculated by

$$p(Z=z|X_i,\theta^{(n)})=\frac{p(X_i|Z=z)p(Z=z|\theta^{(n)})}{\sum_{z'} p(X_i|Z=z')p(Z=z'|\theta^{(n)})}$$

where $p(Z=z|\theta)=\theta_{z_1}\theta_{z_2}$ and $p(X_i|Z=z)=0$ if the genotypes are not consistent with the haplotypes and 1 otherwise.

Note that the numerator is the same quantity you already summed over in `likeG`, and the denominator is that sum. So the E step is `likeG` with the terms kept apart instead of added together.

For the M step the new frequency of a haplotype is its expected count divided by the total expected count:

$$\theta_{z_a}^{(n+1)} = \frac{\sum_i q_i(Z_a=z_a)}{\sum_i \sum_{z_a} q_i(Z_a=z_a)}$$

The posterior is over haplotype *pairs*, so to get the expected count of a single haplotype you sum the pairs it appears in. That is what `countHaplotypes` does.


## Task 7: the E step

`calculateQ` is `likeG` with the 16 terms kept apart and then normalised, so start from your `likeG` and change two lines.

In [ ]:
calculateQ <- function(theta,g1,g2){
  #input: theta and the two genotypes of one individual
  #output: p(Z|X,theta) for the 16 haplotype pairs

  names(theta) <- c("AB","aB","Ab","ab")

  # the numerator p(X|Z)p(Z|theta) for each of the 16 pairs
  pXZ_pZtheta <- c()
  for(z1 in c("AB","aB","Ab","ab"))
    for(z2 in c("AB","aB","Ab","ab")){
        hapToGeno1 <- z1%in%c("AB","Ab") + z2%in%c("AB","Ab")
        hapToGeno2 <- z1%in%c("AB","aB") + z2%in%c("AB","aB")
        hapPair <- paste(z1,z2,sep="/")

        if(g1 != hapToGeno1 | g2 != hapToGeno2)
          pXZ_pZtheta[hapPair] <- 0     # inconsistent with the genotypes
        else
          pXZ_pZtheta[hapPair] <- ????  # p(X|Z)=1 so only p(Z|theta) is left
    }

  # normalise by the sum over all Z, which is the denominator of the E step
  return( ???? )
}

# a check: the posterior has to sum to one, and be flat nowhere the genotypes forbid
q <- calculateQ(theta,1,1)
cat("posterior for a double heterozygote (Aa,Bb):\n")
round(q,4)
cat("\nsums to",sum(q),"\n")

## given: apply calculateQ to all 9 genotype combinations
calculate9Q <- function(theta){
  #input: theta
  #output: p(Z|X,theta) as a 3 x 3 x 16 array. 9 genotype combinations, 16 haplotype pairs
  QforG1G2 <- array(NA,dim=c(3,3,16))
  for(g1 in 0:2)
    for(g2 in 0:2)
      QforG1G2[g1+1,g2+1,] <- calculateQ(theta,g1,g2)
  QforG1G2
}

## Task 8: the M step, and running the EM

The E step is done for you here, using `calculate9Q`. Two lines of the M step are missing. Remember that the posterior is over haplotype *pairs* and `theta` is over single haplotypes, so `countHaplotypes` from Task 3 is what bridges the two.

In [ ]:
emStep <- function(theta,data){

  names(theta) <- c("AB","aB","Ab","ab")
  N <- nrow(data)

  ## E step: look up the posterior for each individual's genotype combination
  Q9 <- calculate9Q(theta)
  QZ <- matrix(0,nrow=nrow(data),ncol=16)
  colnames(QZ) <- names(calculateQ(theta,1,1))  # the 16 haplotype pairs
  for(i in 1:N){
    g1 <- data[i,1]
    g2 <- data[i,2]
    QZ[i,] <- Q9[g1+1,g2+1,]
  }

  ## M step
  # the expected number of each of the 16 haplotype pairs, summed over individuals
  expectedHaploPairs <- ????
  # the expected count of each of the 4 haplotypes
  expectedCountHap <- countHaplotypes(expectedHaploPairs)
  # and the new frequencies
  thetaNew <- ????

  thetaNew
}

cat("\nrun the EM!\n")
thetaEM <- rep(1/4,4)
for(i in 0:10){
  ll <- likelihood(data,thetaEM)
  cat("iter",i,"theta:",thetaEM,"Loglike",ll,"\n")
  thetaEM <- emStep(thetaEM,data)
}

cat("\nthe true haplotype frequencies (which the EM never sees)\n")
freqHap

# compare the last iteration with estFreqHap from Task 3, the estimate you would
# have made if you could see the haplotypes. How close did the EM get?

# Faster version

Again, instead of looping over all 1000 individuals we only need the 9 possible genotype combinations, weighted by their counts.


## Task 9: the same EM step, faster

One line is missing: the 9 genotype combinations weighted by their counts.

In [ ]:
cat("data\n")
(dataTab <- genotypeTab)

emStepFast <- function(theta,dataTab){

  names(theta) <- c("AB","aB","Ab","ab")

  ## E step over the 9 genotype combinations instead of the N individuals
  Q9 <- calculate9Q(theta)
  QZ <- matrix(0,nrow=9,ncol=16)
  colnames(QZ) <- names(calculateQ(theta,1,1))
  i <- 1
  for(g1 in 0:2)
    for(g2 in 0:2){
      QZ[i,] <- ????   # the posterior, weighted by how many individuals have this genotype
      i <- i+1
  }

  ## M step, exactly as before
  expectedHaploPairs <- colSums(QZ)
  expectedCountHap <- countHaplotypes(expectedHaploPairs)
  thetaNew <- expectedCountHap/sum(expectedCountHap)
  thetaNew
}

cat("\nrun the fast EM!\n")
thetaEM <- rep(0.25,4)
for(i in 0:10){
  ll <- likelihoodFast(dataTab,thetaEM)
  cat("iter",i,"theta:",thetaEM,"Loglike",ll,"\n")
  thetaEM <- emStepFast(thetaEM,dataTab)
}

cat("\nthe true haplotype frequencies (which the EM never sees)\n")
freqHap

# the iterations should be identical to Task 8, just quicker

## EM algorithm (proof from class)

If the likelihood is of the form

$log(L(\theta))  =  \sum_i log \left (\sum_j p(X_i,Z_j|\theta) \right )$

and $\sum_j \theta_j=1$

and $p(X|Z,\theta)=p(X|Z)$

and $p(Z_j|\theta)=\theta_j$

then the solution is

### E step
$q_i(Z_j)=p(Z_j|X_i,\theta^{(n)})$

### M step
$\theta_j^{(n+1)} = \frac{\sum_i q_i(Z_j)}{\sum_i \sum_j q_i(Z_j)}$

## Our likelihood

It has the same form.

### log likelihood
$log(L(\theta))=Log(p(X|\theta))=\sum_{i=1}^N log\left(\sum_{z\in\{AB,aB,Ab,ab\}^2}p(X_i|Z=z)p(Z=z|\theta)\right)$

$\theta=(\theta_{AB},\theta_{aB},\theta_{Ab},\theta_{ab})$ so $\sum_{z_a} p(Z_a=z_a|\theta)=1$

and $$p(X_i|Z,\theta)=p(X_i|Z)$$ because the genotypes are directly determined by the pair of haplotypes,

and $$p(Z=z_a|\theta)=\theta_z$$ because the parameter of $z$ is the frequency of $z$.

### E
$q_i(Z=z)=p(Z=z|X_i,\theta^{(n)})$ can be calculated by

$$p(Z=z|X_i,\theta^{(n)})=\frac{p(X_i|Z=z)p(Z=z|\theta^{(n)})}{\sum_{z'} p(X_i|Z=z')p(Z=z'|\theta^{(n)})}$$

where $p(Z=z|\theta)=\theta_{z_1}\theta_{z_2}$ and $p(X_i|Z=z)=0$ if the genotypes are not consistent with the haplotypes and 1 otherwise.

For a single haplotype instead of a pair

$$p(Z_a=z_a|X_i,\theta^{(n)})=\sum_{z'}\left(p(Z=(z_a,z')|X_i,\theta^{(n)}) + p(Z=(z',z_a)|X_i,\theta^{(n)})\right)$$

### M
$\theta_{z_a}^{(n+1)} = \frac{\sum_i q_i(Z_a=z_a)}{\sum_i \sum_j q_i(Z_a=z_a)}$